## Model Configuration

In [1]:
import os
import torch

# Set environment variables to help manage CUDA memory
os.environ['CUDA_VISIBLE_DEVICES'] = "MIG-b5356651-0d8e-5cd1-bdf3-ccbb8b221031"
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"Using device: {torch.cuda.get_device_name(0)}")

Using device: NVIDIA A100 80GB PCIe MIG 3g.40gb


### UNet Model for Autoregressive Rollout

In [ ]:
import torch
import torch.nn as nn

def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )


class Encoder(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(Encoder, self).__init__()
        self.conv_block = conv_block(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.conv_block(x)
        x_pooled = self.pool(x)
        return x, x_pooled


class Decoder(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(Decoder, self).__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv_block = conv_block(in_channels, out_channels)

    def forward(self, x, skip_connection):
        x = self.up(x)
        x = torch.cat([x, skip_connection], dim=1)
        x = self.conv_block(x)
        return x


class UNet(nn.Module):
    def __init__(self, in_channels=6, out_channels=1):
        super(UNet, self).__init__()

        self.enc1 = Encoder(in_channels, 64)
        self.enc2 = Encoder(64, 128)
        self.enc3 = Encoder(128, 256)
        self.enc4 = Encoder(256, 512)

        self.bottleneck2 = conv_block(512, 1024)

        self.dec4 = Decoder(1024, 512)
        self.dec3 = Decoder(512, 256)
        self.dec2 = Decoder(256, 128)
        self.dec1 = Decoder(128, 64)

        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):

        # Encoder path
        x1, x1_pooled = self.enc1(x)
        x2, x2_pooled = self.enc2(x1_pooled)
        x3, x3_pooled = self.enc3(x2_pooled)
        x4, x4_pooled = self.enc4(x3_pooled)

        # Bottleneck
        x_bottleneck = self.bottleneck2(x4_pooled)

        # Decoder path
        x = self.dec4(x_bottleneck, x4)
        x = self.dec3(x, x3)
        x = self.dec2(x, x2)
        x = self.dec1(x, x1)

        # Final convolution
        x = self.final_conv(x)
        return x


### Test Model Configuration

In [ ]:
# initialize dummy input
X = torch.randn((2, 6, 256, 256))

# initialize model
m = UNet(in_channels=6, out_channels=1)

# forward pass
y = m(X); print(y.shape) # torch.Size([2, 1, 256, 256])


torch.Size([2, 1, 256, 256])


### SFNO Model for Autoregressive Rollout

The **Spherical Fourier Neural Operator (SFNO)** is a state-of-the-art operator learning architecture designed for spherical domains like Earth's atmosphere.

### Architecture Overview

The model processes input `(B, 6, 90, 144)` through three main stages:

#### A. Lifting (Embedding)
- **Input**: 6 physical channels (u, v, ω, prec, BCB_biomass_src, BCB)
- **Operation**: Pixel-wise convolution projects variables into latent space
- **Output**: `embed_dim=16` channels
- **Purpose**: Learn complex relationships between physical variables before spatial mixing

#### B. Spherical Fourier Layers (Core Processing)
With `num_layers=4`, each layer performs a "sandwich" operation:

1. **Transform to Spectral Domain**
    - Converts data from Physical Grid (lat/lon) → Spectral Domain (wavenumbers)
    - Uses Spherical Harmonic Transform (SHT) instead of FFT
    - **Key advantage**: Handles the "pole problem" where lat/lon grid points cluster at poles, respecting Earth's spherical geometry

2. **Spectral Convolution (Learnable Operator)**
    - Multiplies spectral components by learnable complex weights: $Z_{spectral} = R \cdot X_{spectral}$
    - **Effect**: Dampens, amplifies, or phase-shifts global weather patterns
    - Acts as a soft differential equation solver for large-scale dynamics

3. **Inverse Transform**
    - Spectral Domain → Physical Domain via inverse SHT
    - Applies non-linearity (GELU) and MLP in physical domain
    - Captures local, high-frequency phenomena (turbulence, sharp fronts)

#### C. Projection (Head)
- **Input**: 16 latent channels
- **Operation**: Linear projection
- **Output**: 1 channel (BC concentration prediction)

### Key Advantages over Standard FNO
- **Spherical geometry awareness**: Proper handling of Earth's curvature
- **No pole distortion**: SHT naturally respects spherical domain properties
- **Global pattern learning**: Spectral convolution learns atmospheric dynamics at all scales


In [2]:
import torch
from sfno import SphericalFourierNeuralOperator

model = SphericalFourierNeuralOperator(
    img_size=(90, 144),
    grid="equiangular",
    num_layers=4,
    scale_factor=6,
    embed_dim=16,
    residual_prediction=False,  # <--- MUST be False to change channel count
    pos_embed="spectral",
    use_mlp=True,
    normalization_layer="none",
    in_chans=6,   # Input takes 6 channels
    out_chans=1   # Output produces 1 channel
)

X = torch.randn((2, 6, 90, 144)) 
y = model(X)
print(y.shape)
# Output will be: torch.Size([2, 1, 90, 144])

torch.Size([2, 1, 90, 144])


## Data Preparation and Loading

### Padding Module

In [2]:
import json

import numpy as np
import xarray as xr

import torch
from torch.utils.data import Dataset, DataLoader

class E33OMAPAD(Dataset):

    def __init__(self, padding):
        super(E33OMAPAD, self).__init__()

        self.padding = padding

    def _padding_data(self, data):
        """
        Apply cyclic padding along the longitude (W dimension)
        and reflective padding along the latitude (H dimension).
        """
        pad_lat = self.padding[0]  # top and bottom
        pad_lon = self.padding[1]  # left and right

        H = data.shape[-2]  # latitude
        W = data.shape[-1]  # longitude

        # --- Cyclic padding along longitude (axis=4) ---
        if (pad_lon <= W):
            data = np.concatenate([data[..., -pad_lon:], data, data[..., :pad_lon]], axis=-1)
        else:
            raise AttributeError(
                f"Requested longitude padding ({pad_lon}) exceeds image width ({W})."
            )

        # --- Reflective padding along latitude (axis=3) ---
        pad_top = pad_lat
        pad_bottom = pad_lat

        if (pad_top <= H) and (pad_bottom <= H):
            top = np.flip(data[..., 1:pad_top+1, :], axis=-2)
            bottom = np.flip(data[..., -pad_bottom-1:-1, :], axis=-2)
            data = np.concatenate((top, data, bottom), axis=-2)
        else:
            raise AttributeError(
                f"Requested latitude padding ({pad_top}, {pad_bottom}) exceeds image height ({H})."
            )

        return data

    def _get_data_statistics(self):

        with open('/home/serfani/projects/nasa-niswan/statistics/1950.json', 'r') as jf:
            data = json.load(jf)
            data = data['set1']
        
        X1_mean = data['u']['mean']
        X2_mean = data['v']['mean']
        X3_mean = data['w']['mean']
        X4_mean = data['prec']['mean']

        X1_std = data['u']['std']
        X2_std = data['v']['std']
        X3_std = data['w']['std']
        X4_std = data['prec']['std']

        X5_mean = data['bc_src']['mean']; X5_std = data['bc_src']['std']
        self.y_mean = data['bc_conc']['mean']; self.y_std = data['bc_conc']['std']
        
        self.X_mean = np.stack([X1_mean, X2_mean, X3_mean, X4_mean, X5_mean, self.y_mean], axis=0, dtype=np.float32).reshape(1, -1, 1, 1)
        self.X_std = np.stack([X1_std, X2_std, X3_std, X4_std, X5_std, self.y_std], axis=0, dtype=np.float32).reshape(1, -1, 1, 1)

### Dataset Module

In [3]:
class E33OMA(E33OMAPAD):

    def __init__(self, padding, period='train', rollout_steps=1, root='/home/serfani/serfani_data1/E3OMA1950.zarr'):
        super(E33OMA, self).__init__(padding)
        
        self.root = root
        self.period = period
        self.rollout_steps = rollout_steps

        # Open as a single lazy object
        self.ds = xr.open_zarr(self.root)

        self.times = self.ds.time.values
        self.lat = self.ds.lat.values
        self.lon = self.ds.lon.values

        self._get_data_index()
        self._get_data_statistics()

    def _get_data_index(self):

        X_timestamps = self.times[:-1]
        y_timestamps = self.times[1:]

        # Create sequences
        X_datetimeindex = self.create_sequences(X_timestamps)
        y_datetimeindex = self.create_sequences(y_timestamps)

        idx = np.arange((len(self.times) * 0.5) - self.rollout_steps, dtype='int')  # The entire year of 1950
        rng = np.random.default_rng(42)
        rng.shuffle(idx)

        # Calculate the split point (ensuring integer)
        split_point = int(0.9 * len(idx))  # Takes the floor automatically for non-integers

        if self.period == 'train':
            self.X_datetimeindex = X_datetimeindex[idx[:split_point]]
            self.y_datetimeindex = y_datetimeindex[idx[:split_point]]
        
        elif self.period == 'val':
            self.X_datetimeindex = X_datetimeindex[idx[split_point:]] 
            self.y_datetimeindex = y_datetimeindex[idx[split_point:]]
        
        if self.period == 'test':
            self.X_datetimeindex = X_datetimeindex[len(idx):]
            self.y_datetimeindex = y_datetimeindex[len(idx):]
    
    def create_sequences(self, data_array):
        return np.array([data_array[i:i+self.rollout_steps] for i in range(len(data_array)-self.rollout_steps+1)])


    def __getitem__(self, index):

        # Get timesteps for this sample
        X_timesteps = self.X_datetimeindex[index]
        y_timesteps = self.y_datetimeindex[index]

        X = self.ds[['u', 'v', 'omega', 'prec', 'BCB_biomass_src', 'BCB']].sel(time=X_timesteps).to_array().values.transpose(1, 0, 2, 3)
        y = self.ds[['BCB']].sel(time=y_timesteps).to_array().values.transpose(1, 0, 2, 3)
        
        X = (X - self.X_mean) / self.X_std
        y = (y - self.y_mean) / self.y_std

        if self.padding != [0, 0]:
            X = self._padding_data(X) # (5, seq_len, 90 + (2xpadding), 144 + (2xpadding))

        X = torch.from_numpy(X).type(torch.float32) # torch image: (channels, sequence_length, height, width)
        y = torch.from_numpy(y).type(torch.float32) # torch image: (channels, sequence_length, height, width)

        return X, y 
         
    def __len__(self):
        return len(self.X_datetimeindex)

### Test Dataset Module

In [15]:
dataset = E33OMA(period='train', padding=[83, 56], rollout_steps=3)
dataloader = DataLoader(dataset, batch_size=2, num_workers=1, shuffle=True)

dataiter = iter(dataloader)
X, y = next(dataiter)

print(len(dataset))
print(X.shape, y.shape)

15765
torch.Size([2, 3, 6, 256, 256]) torch.Size([2, 3, 1, 90, 144])


In [21]:
def _apply_padding(data, padding):
    """
    Apply cyclic padding along the longitude (W dimension)
    and reflective padding along the latitude (H dimension).
    """
    pad_lat = padding[0]  # top and bottom
    pad_lon = padding[1]  # left and right

    H = data.shape[-2]  # latitude
    W = data.shape[-1]  # longitude

    # --- Cyclic padding along longitude (axis=4) ---
    if (pad_lon <= W):
        data = torch.concatenate([data[..., -pad_lon:], data, data[..., :pad_lon]], axis=-1)
    else:
        raise AttributeError(
            f"Requested longitude padding ({pad_lon}) exceeds image width ({W})."
        )

    # --- Reflective padding along latitude (axis=3) ---
    pad_top = pad_lat
    pad_bottom = pad_lat

    if (pad_top <= H) and (pad_bottom <= H):
        top = torch.flip(data[..., 1:pad_top+1, :], dims=[-2])
        bottom = torch.flip(data[..., -pad_bottom-1:-1, :], dims=[-2])
        data = torch.concatenate((top, data, bottom), axis=-2)
    else:
        raise AttributeError(
            f"Requested latitude padding ({pad_top}, {pad_bottom}) exceeds image height ({H})."
        )

    return data

In [24]:
print(_apply_padding(torch.randn((2, 6, 90, 144)), padding=[83, 56]).shape)

torch.Size([2, 6, 256, 256])


In [ ]:
import csv
from tqdm import tqdm
import torch.optim as optim


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"Using device: {torch.cuda.get_device_name(0)}")

# Config
MAX_ROLLOUT = 48
IMAGE_SIZE = [90, 144] 
PADDING = [83, 56] # [Pad_Y, Pad_X]

# Model (Assuming UNet is imported or defined)
model = UNet(in_channels=6, out_channels=1).to(device)

# Dataset & Loader (Assuming E33OMA is imported or defined)
dataset_train = E33OMA(period='train', padding=PADDING, rollout_steps=MAX_ROLLOUT) 
dataset_val = E33OMA(period='val', padding=PADDING, rollout_steps=MAX_ROLLOUT)

train_loader = DataLoader(dataset_train, batch_size=8, shuffle=True)
val_loader = DataLoader(dataset_val, batch_size=8, shuffle=False)

# Loss Functions
criterion_mse = nn.MSELoss()
criterion_mae = nn.L1Loss()

# Training Schedule Config
TEACHER_FORCING_EPOCHS = 30
WARMUP_EPOCHS = 10
COSINE_EPOCHS = 20
PEAK_LR = 1e-3
BASE_LR = 1e-4

best_val_loss = float('inf')

# ==========================================
# 2. Optimizer & Schedulers
# ==========================================

optimizer = optim.Adam(model.parameters(), lr=PEAK_LR)

# Phase 1: Linear Warmup (Epochs 0-9)
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=BASE_LR/PEAK_LR, end_factor=1.0, total_iters=WARMUP_EPOCHS
)

# Phase 2: Cosine Annealing (Epochs 10-29)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=COSINE_EPOCHS, eta_min=BASE_LR
)

# Phase 3: Constant (Epochs 30+)
constant_scheduler = torch.optim.lr_scheduler.ConstantLR(
    optimizer, factor=BASE_LR/PEAK_LR, total_iters=200 
)

scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler, constant_scheduler],
    milestones=[WARMUP_EPOCHS, TEACHER_FORCING_EPOCHS]
)

# ==========================================
# 3. Training Loop
# ==========================================

TOTAL_EPOCHS = TEACHER_FORCING_EPOCHS + (MAX_ROLLOUT - 1)

print(f"Starting training for {TOTAL_EPOCHS} epochs...")

for epoch in range(TOTAL_EPOCHS):
    
    # --- Determine Phase & Rollout Steps ---
    if epoch < TEACHER_FORCING_EPOCHS:
        rollout_steps = 1
        phase_name = "Teacher Forcing"
    else:
        rollout_steps = (epoch - TEACHER_FORCING_EPOCHS) + 2
        rollout_steps = min(rollout_steps, MAX_ROLLOUT)
        phase_name = f"Curriculum ({rollout_steps} steps)"
    
    # --- Training Step ---
    model.train()
    train_loss_accum = 0.0
    
    # Wrap train_loader with tqdm
    train_loop = tqdm(train_loader, desc=f"Ep [{epoch+1}/{TOTAL_EPOCHS}] {phase_name}", leave=True)
    
    for batch_idx, (inputs, targets) in enumerate(train_loop):
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        loss = 0
        
        current_input = inputs[:, 0]
        
        for k in range(rollout_steps):
            # Forward
            pred_full = model(current_input)
            
            # Crop
            pred_cropped = pred_full[..., 
                                     PADDING[0]:PADDING[0]+IMAGE_SIZE[0], 
                                     PADDING[1]:PADDING[1]+IMAGE_SIZE[1]]

            # Loss
            gt_step = targets[:, k] 
            loss += criterion_mse(pred_cropped, gt_step) + criterion_mae(pred_cropped, gt_step)
            
            # Autoregression Setup
            if k < rollout_steps - 1:
                pred_padded = _apply_padding(pred_cropped)
                current_input = inputs[:, k + 1].clone()
                current_input[:, -1] = pred_padded.squeeze(1) 

        # Normalize Loss
        loss = loss / rollout_steps
        loss.backward()
        optimizer.step()
        
        train_loss_accum += loss.item()
        
        # Update tqdm bar with the current batch loss
        train_loop.set_postfix(batch_loss=f"{loss.item():.4f}")

    # --- Step Scheduler ---
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    avg_train_loss = train_loss_accum / len(train_loader)
    
    # --- Validation Step ---
    model.eval()
    val_loss_accum = 0.0
    
    # Wrap val_loader with tqdm (optional, but good for consistency)
    val_loop = tqdm(val_loader, desc="Validation", leave=False)
    
    with torch.no_grad():
        for inputs, targets in val_loop:
            inputs = inputs.to(device)
            targets = targets.to(device)
            loss = 0
            
            current_input = inputs[:, 0]
            
            for k in range(rollout_steps):
                pred_full = model(current_input)
                
                pred_cropped = pred_full[..., 
                                         PADDING[0]:PADDING[0]+IMAGE_SIZE[0], 
                                         PADDING[1]:PADDING[1]+IMAGE_SIZE[1]]

                gt_step = targets[:, k]
                loss += criterion_mse(pred_cropped, gt_step) + criterion_mae(pred_cropped, gt_step)
                
                if k < rollout_steps - 1:
                    pred_padded = _apply_padding(pred_cropped)
                    current_input = inputs[:, k + 1].clone()
                    current_input[:, -1] = pred_padded.squeeze(1)
            
            val_loss_accum += (loss / rollout_steps).item()
            
    avg_val_loss = val_loss_accum / len(val_loader)
    
    # --- End of Epoch Summary ---
    # We use tqdm.write() to ensure the summary prints cleanly above the progress bars
    tqdm.write(f"--> Summary Ep {epoch+1}: LR: {current_lr:.6f} | "
               f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # --- Logging to CSV ---
    with open("./01-16-2026/training_log.csv", mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([epoch + 1, rollout_steps, current_lr, avg_train_loss, avg_val_loss])
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "./01-16-2026/best_model.pth")
        tqdm.write(f"    [Saved Best Model] Val Loss: {best_val_loss:.4f}")

In [6]:
class LatitudeWeightedMSE(nn.Module):
    def __init__(self, lat_res=90, device="cuda"):
        super().__init__()
        self.device = device
        
        # 1. Generate Latitude weights (Cosine of latitude)
        # Grid assumes typically [90, -90] or [-90, 90]. 
        # For Equiangular grid, we create a 1D tensor of weights.
        
        # Create a linear space for latitude from -90 to 90 (in radians)
        lats = torch.linspace(-torch.pi/2, torch.pi/2, lat_res)
        
        # Calculate cosine weights
        weights = torch.cos(lats)
        
        # Reshape to broadcast: (1, 1, Lat, 1) -> fits (B, C, H, W)
        # We assume H=lat_res
        self.weights = weights.view(1, 1, -1, 1).to(device)

    def forward(self, pred, target):
        # Squared Error
        diff_sq = (pred - target) ** 2
        
        # Weighted Mean
        # We multiply error by weights, then mean.
        weighted_mse = torch.mean(diff_sq * self.weights)
        
        return weighted_mse


In [ ]:
import csv
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sfno import SphericalFourierNeuralOperator
# from your_dataset_module import E33OMA # Uncomment this

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"Using device: {torch.cuda.get_device_name(0)}")

# Config
MAX_ROLLOUT = 48
IMAGE_SIZE = [90, 144] 
PADDING = [0, 0]

# Model
model = SphericalFourierNeuralOperator(
    img_size=(90, 144),
    grid="equiangular",
    num_layers=4,
    scale_factor=6,
    embed_dim=16,
    residual_prediction=False, # DISABLED: We do manual residual
    pos_embed="spectral",
    use_mlp=True,
    normalization_layer="none",
    in_chans=6,   # Input: 1 BCB + 5 Weather
    out_chans=1   # Output: 1 Delta BCB
).to(device)

# Dataset & Loader (Assuming E33OMA is imported)
dataset_train = E33OMA(period='train', padding=PADDING, rollout_steps=MAX_ROLLOUT) 
dataset_val = E33OMA(period='val', padding=PADDING, rollout_steps=MAX_ROLLOUT)

train_loader = DataLoader(dataset_train, batch_size=8, shuffle=True)
val_loader = DataLoader(dataset_val, batch_size=8, shuffle=False)

# Loss Functions
criterion = LatitudeWeightedMSE(lat_res=90, device=device)

# Training Schedule
TEACHER_FORCING_EPOCHS = 30
WARMUP_EPOCHS = 10
COSINE_EPOCHS = 20
PEAK_LR = 1e-3
BASE_LR = 1e-4

best_val_loss = float('inf')

optimizer = optim.Adam(model.parameters(), lr=PEAK_LR)

# Phase 1: Linear Warmup
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=BASE_LR/PEAK_LR, end_factor=1.0, total_iters=WARMUP_EPOCHS
)
# Phase 2: Cosine Annealing
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=COSINE_EPOCHS, eta_min=BASE_LR
)
# Phase 3: Constant
constant_scheduler = torch.optim.lr_scheduler.ConstantLR(
    optimizer, factor=BASE_LR/PEAK_LR, total_iters=200 
)

scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler, constant_scheduler],
    milestones=[WARMUP_EPOCHS, TEACHER_FORCING_EPOCHS]
)

# ==========================================
# 3. Training Loop
# ==========================================

TOTAL_EPOCHS = TEACHER_FORCING_EPOCHS + (MAX_ROLLOUT - 1)
print(f"Starting training for {TOTAL_EPOCHS} epochs...")

for epoch in range(TOTAL_EPOCHS):
    
    # --- Determine Phase & Rollout Steps ---
    if epoch < TEACHER_FORCING_EPOCHS:
        rollout_steps = 1
        phase_name = "Teacher Forcing"
    else:
        rollout_steps = (epoch - TEACHER_FORCING_EPOCHS) + 2
        rollout_steps = min(rollout_steps, MAX_ROLLOUT)
        phase_name = f"Curriculum ({rollout_steps} steps)"
    
    # --- Training Step ---
    model.train()
    train_loss_accum = 0.0
    
    train_loop = tqdm(train_loader, desc=f"Ep [{epoch+1}/{TOTAL_EPOCHS}] {phase_name}", leave=True)
    
    for batch_idx, (inputs, targets) in enumerate(train_loop):
        inputs = inputs.to(device) # Shape: (B, Time, 6, H, W) -> 6 chans = [BCB, Weather...]
        targets = targets.to(device) # Shape: (B, Time, 1, H, W) -> Ground Truth BCB
        
        optimizer.zero_grad()
        loss = 0
        
        # Initialize State with T=0
        # We assume Channel 0 is BCB, Channels 1-5 are Weather
        current_bcb = inputs[:, 0, -1:] # (B, 1, H, W) - The BCB at T=0
        
        for k in range(rollout_steps):
            # 1. Prepare Input
            # We need the Weather at T=k (the current step driving the change)
            # inputs[:, k, 1:] extracts channels 1-5 (Weather)
            current_weather = inputs[:, k, :-1] 
            
            # Stack: (B, 1, H, W) + (B, 5, H, W) -> (B, 6, H, W)
            model_input = torch.cat([current_weather, current_bcb], dim=1)

            # 2. Forward Pass (Predict Delta)
            delta_bcb = model(model_input)
            
            # 3. Manual Residual Connection (Physics Step)
            # New State = Old State + Change
            pred_bcb = current_bcb + delta_bcb

            # 4. Calculate Loss
            # Compare Predicted State vs Ground Truth State at T=k
            gt_bcb = targets[:, k] # The actual BCB at this step
            loss += criterion(pred_bcb, gt_bcb)
            
            # 5. Update State for Next Step (Autoregressive)
            # Crucial: We must detach() if we don't want gradients to flow infinitely back
            # (Though for training we usually keep graph connected for BPTT)
            # In curriculum learning, we usually feed the PREDICTION into the next step
            
            if k < rollout_steps - 1:
                current_bcb = pred_bcb # Feed the predicted state into the next iteration
        
        # Normalize Loss
        loss = loss / rollout_steps
        loss.backward()
        optimizer.step()
        
        train_loss_accum += loss.item()
        train_loop.set_postfix(batch_loss=f"{loss.item():.4f}")

    # --- Step Scheduler ---
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    avg_train_loss = train_loss_accum / len(train_loader)
    
    # --- Validation Step ---
    model.eval()
    val_loss_accum = 0.0
    val_loop = tqdm(val_loader, desc="Validation", leave=False)
    
    with torch.no_grad():
        for inputs, targets in val_loop:
            inputs = inputs.to(device)
            targets = targets.to(device)
            loss = 0
            
            # Reset State for Validation Batch
            current_bcb = inputs[:, 0, -1:]
            
            for k in range(rollout_steps):
                current_weather = inputs[:, k, :-1]
                model_input = torch.cat([current_weather, current_bcb], dim=1)

                delta_bcb = model(model_input)
                
                # Manual Residual Update
                pred_bcb = current_bcb + delta_bcb
                
                gt_bcb = targets[:, k]
                loss += criterion(pred_bcb, gt_bcb)

                if k < rollout_steps - 1:
                    current_bcb = pred_bcb
            
            val_loss_accum += (loss / rollout_steps).item()
            
    avg_val_loss = val_loss_accum / len(val_loader)
    
    # --- Logging ---
    tqdm.write(f"--> Summary Ep {epoch+1}: LR: {current_lr:.6f} | "
               f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    with open("./01-16-2026/training_log.csv", mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([epoch + 1, rollout_steps, current_lr, avg_train_loss, avg_val_loss])
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "./01-16-2026/best_model.pth")
        tqdm.write(f"    [Saved Best Model] Val Loss: {best_val_loss:.4f}")